# Cloud Native Geospatial Capstone Project

## Project Goals

### Summary

Create a file collection summary (e.g. number of datapoints / deployments)  by applying SQL commands to any parquet data set stored in `aodn-cloud-optimised` using DuckDB. (I use dask and S3 as well) 

### Test collection

slocum_glider_delayed_qc.parquet/ -> 553 objects, ~34.2GB 

### Motivation

Using familiar tools such as xarray and pandas are slow!



Import packages

In [1]:
#!source /home/mphemming/Documents/Projects/Other/thriveGeo/thriveGeo/bin/activate

In [2]:
import duckdb
import dask.dataframe as dd
import s3fs
import botocore.session
import sys
import os
from io import StringIO
from datetime import datetime
import time
import matplotlib.pyplot as plt
import cartopy.feature as cfeature
import cartopy.crs as ccrs
import io
import base64

# for timing the notebook execution
notebook_start = time.perf_counter()

# load AWS profile name
with open("AWS.txt", "r", encoding="utf-8") as f:
    aws_profile = f.read().strip()
    

In [3]:
# Define the dataset name and S3 URI

dataset_name = 'vessel_co2_delayed_qc'
s3_uri = f's3://imos-optimised-nonproduction/{dataset_name}.parquet/'

In [4]:
# get a list of all parquet datasets on S3

def list_parquet_files():
    fs = s3fs.S3FileSystem(anon=False)
    files = fs.glob(f's3://imos-optimised-nonproduction/*.parquet')
    return files

available_parquet_datasets = list_parquet_files()
available_parquet_datasets

['imos-optimised-nonproduction/aggregated_amsa_nonqc.parquet',
 'imos-optimised-nonproduction/aggregated_dugong_nonqc.parquet',
 'imos-optimised-nonproduction/aggregated_kelp_nonqc.parquet',
 'imos-optimised-nonproduction/aggregated_seabird_nonqc.parquet',
 'imos-optimised-nonproduction/aggregated_seagrass_nonqc.parquet',
 'imos-optimised-nonproduction/animal_ctd_satellite_relay_tagging_delayed_qc.parquet',
 'imos-optimised-nonproduction/diver_photoquadrat_score_qc.parquet',
 'imos-optimised-nonproduction/mooring_acidification_delayed_qc.parquet',
 'imos-optimised-nonproduction/mooring_ctd_delayed_qc.parquet',
 'imos-optimised-nonproduction/mooring_hourly_timeseries_delayed_qc.parquet',
 'imos-optimised-nonproduction/mooring_satellite_altimetry_calibration_validation.parquet',
 'imos-optimised-nonproduction/mooring_timeseries_realtime_qc.parquet',
 'imos-optimised-nonproduction/slocum_glider_delayed_qc.parquet',
 'imos-optimised-nonproduction/vessel_air_sea_flux_product_delayed.parquet

### Get a summary of the data set using S3 and dask


In [5]:
# get s3 size of dataset and number of files
def get_s3_size_and_file_count(s3_uri):
    fs = s3fs.S3FileSystem(anon=False)
    total_size = 0
    file_count = 0
    
    files = fs.glob(s3_uri + '**/*.parquet')

    for file in files:
        total_size += fs.info(file)['Size']
        file_count += 1
        
    print(f"Total size of dataset: {total_size / (1024**3):.2f} GB")
    print(f"Total number of files: {file_count}")

    return total_size, file_count, files

_,_,files = get_s3_size_and_file_count(s3_uri)

files

Total size of dataset: 0.22 GB
Total number of files: 2704


['imos-optimised-nonproduction/vessel_co2_delayed_qc.parquet/timestamp=1199145600/polygon=01030000000100000005000000000000000080614000000000000049C00000000000C0624000000000000049C00000000000C0624000000000000044C0000000000080614000000000000044C0000000000080614000000000000049C0/platform_code=VLHJ/IMOS_SOOP-CO2_GST_20080111T093235Z_VLHJ_FV01.nc-0.parquet',
 'imos-optimised-nonproduction/vessel_co2_delayed_qc.parquet/timestamp=1199145600/polygon=01030000000100000005000000000000000080614000000000008046C00000000000C0624000000000008046C00000000000C0624000000000008041C0000000000080614000000000008041C0000000000080614000000000008046C0/platform_code=VLHJ/IMOS_SOOP-CO2_GST_20080111T093235Z_VLHJ_FV01.nc-0.parquet',
 'imos-optimised-nonproduction/vessel_co2_delayed_qc.parquet/timestamp=1199145600/polygon=010300000001000000050000000000000000E0604000000000000049C0000000000020624000000000000049C0000000000020624000000000000044C00000000000E0604000000000000044C00000000000E0604000000000000049C0/platform_co

In [6]:
# extract NetCDF files appended to the parquet dataset

nc_files = [
    os.path.basename(p).split(".nc")[0] + ".nc"
    for p in files
]
nc_files 

['IMOS_SOOP-CO2_GST_20080111T093235Z_VLHJ_FV01.nc',
 'IMOS_SOOP-CO2_GST_20080111T093235Z_VLHJ_FV01.nc',
 'IMOS_SOOP-CO2_GST_20080111T093235Z_VLHJ_FV01.nc',
 'IMOS_SOOP-CO2_GST_20080228T083851Z_VLHJ_FV01.nc',
 'IMOS_SOOP-CO2_GST_20080204T030605Z_VLHJ_FV01.nc',
 'IMOS_SOOP-CO2_GST_20080228T083851Z_VLHJ_FV01.nc',
 'IMOS_SOOP-CO2_GST_20080204T030605Z_VLHJ_FV01.nc',
 'IMOS_SOOP-CO2_GST_20080204T030605Z_VLHJ_FV01.nc',
 'IMOS_SOOP-CO2_GST_20080204T030605Z_VLHJ_FV01.nc',
 'IMOS_SOOP-CO2_GST_20080204T030605Z_VLHJ_FV01.nc',
 'IMOS_SOOP-CO2_GST_20080321T061136Z_VLHJ_FV01.nc',
 'IMOS_SOOP-CO2_GST_20080321T061136Z_VLHJ_FV01.nc',
 'IMOS_SOOP-CO2_GST_20080228T083851Z_VLHJ_FV01.nc',
 'IMOS_SOOP-CO2_GST_20080228T083851Z_VLHJ_FV01.nc',
 'IMOS_SOOP-CO2_GST_20080228T083851Z_VLHJ_FV01.nc',
 'IMOS_SOOP-CO2_GST_20080321T061136Z_VLHJ_FV01.nc',
 'IMOS_SOOP-CO2_GST_20080228T083851Z_VLHJ_FV01.nc',
 'IMOS_SOOP-CO2_GST_20080228T083851Z_VLHJ_FV01.nc',
 'IMOS_SOOP-CO2_GST_20080321T061136Z_VLHJ_FV01.nc',
 'IMOS_SOOP-

In [7]:
from pathlib import Path
p = Path("/home/mphemming/.aws/sso/cache")
print("cache dir exists:", p.exists())
print("cache dir readable:", os.access(p, os.R_OK))
print("cache dir executable:", os.access(p, os.X_OK))

for f in p.glob("*.json"):
    print(f, "readable:", os.access(f, os.R_OK))

cache dir exists: True
cache dir readable: True
cache dir executable: True
/home/mphemming/.aws/sso/cache/8d2ca06b7afd6d468ab38b113ba2bb12ca36cdda.json readable: True
/home/mphemming/.aws/sso/cache/2736fab291f04e69b62d490c3c09361f5b82461a.json readable: True


In [8]:
# get the date the parquet dataset was created and last updated
# Need to login via SSO

fs = s3fs.S3FileSystem()
    
latest = None
first = None

for path in fs.find(s3_uri):
    info = fs.info(path)
    modified = info["LastModified"]
    
    if latest is None or modified > latest:
        latest = modified
    if first is None or modified < first:
        first = modified
        

In [9]:
def summarise_ds(s3_uri):
    # use dask to read the parquet file from S3
    ddf = dd.read_parquet(s3_uri, engine='pyarrow',
                        storage_options={"profile": aws_profile})
    for col, dtype in ddf.dtypes.items():
        print(f"{col:<35} {dtype}")
    
    # summarise partitions
    print(' ' * 150)
    print(f"This data set has {ddf.npartitions} partitions.")
    print(' ' * 150)
    fs = s3fs.S3FileSystem(profile=aws_profile)
    partitions = fs.ls(s3_uri)
    partition_names = [p.split("=")[-1] for p in partitions]
    # get partition and non-partition columns
    partition_column_names = list(
        ddf.select_dtypes(include="category").columns
    )
    column_names = list(
        ddf.select_dtypes(exclude="category").columns
    )
    print(f"Partition column names: {', '.join(partition_column_names)}")

    return partition_names, partition_column_names, column_names  


partition_names, partition_column_names, column_names = summarise_ds(s3_uri)
    

PSAL                                float64
TEMP                                float64
TIME                                datetime64[ns]
TYPE                                string
WDIR                                float64
WSPD                                float64
DfCO2                               float64
TEMP_2                              float64
H2OFLOW                             float64
SUBFLAG                             float32
LATITUDE                            float64
filename                            string
LICORflow                           float64
LONGITUDE                           float64
Press_ATM                           float64
cruise_id                           string
xCO2EQ_PPM                          float64
Press_Equil                         float64
fCO2SW_UATM                         float64
vessel_name                         string
xCO2ATM_PPM                         float64
PSAL_quality_control                float32
TEMP_quality_control         

### Use DuckDB to get information

In [10]:
# Initialize DuckDB connection and setup

# Ensure DuckDB and botocore resolve the same AWS profile.
os.environ["AWS_PROFILE"] = aws_profile

con = duckdb.connect()

con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute("SET s3_region='ap-southeast-2';")

# Pull credentials from botocore for the selected profile (supports SSO/session tokens)
# and pass them explicitly to DuckDB.
bc_session = botocore.session.Session(profile=aws_profile)
bc_creds = bc_session.get_credentials()
if bc_creds is None:
    raise RuntimeError(
        f"No AWS credentials found for profile '{aws_profile}'. Run: aws sso login --profile {aws_profile}"
    )

frozen = bc_creds.get_frozen_credentials()

def _esc(v):
    return v.replace("'", "''")

con.execute(f"SET s3_access_key_id='{_esc(frozen.access_key)}';")
con.execute(f"SET s3_secret_access_key='{_esc(frozen.secret_key)}';")
if frozen.token:
    con.execute(f"SET s3_session_token='{_esc(frozen.token)}';")

con.execute("PRAGMA enable_progress_bar") # show progress bar for long-running queries
con.execute("PRAGMA threads=14") # use all available threads for query execution

In [11]:
# get list of all parquet files in the dataset via DuckDB
paths = con.sql(f"""
    SELECT 
        file
    FROM glob('{s3_uri}**/*.parquet')
""").df()

paths["file"]

0       s3://imos-optimised-nonproduction/vessel_co2_d...
1       s3://imos-optimised-nonproduction/vessel_co2_d...
2       s3://imos-optimised-nonproduction/vessel_co2_d...
3       s3://imos-optimised-nonproduction/vessel_co2_d...
4       s3://imos-optimised-nonproduction/vessel_co2_d...
                              ...                        
2699    s3://imos-optimised-nonproduction/vessel_co2_d...
2700    s3://imos-optimised-nonproduction/vessel_co2_d...
2701    s3://imos-optimised-nonproduction/vessel_co2_d...
2702    s3://imos-optimised-nonproduction/vessel_co2_d...
2703    s3://imos-optimised-nonproduction/vessel_co2_d...
Name: file, Length: 2704, dtype: object

In [12]:
# Define SQL query for DuckDB

partition_names_sql = ", ".join(f"'{d}'" for d in partition_names[1::]) # skip the first partition which is the metadata file

# grab the first partition column name (e.g. deployment_code for slocum underwater gliders)
partition_column_name = partition_column_names[0]

# If the dataset has TIME, LONGITUDE and LATITUDE columns, assume it's an IMOS dataset and get more detailed summary statistics.
# Otherwise, just get row counts by partition.
if 'TIME' in column_names and 'LONGITUDE' in column_names and 'LATITUDE' in column_names:

    # This is an IMOS dataset (e.g. slocum_glider_delayed_qc)
    # Variable names are standardised

    sql = f"""
    SELECT
    {partition_column_name},
    MIN(TIME) AS start_time,
    MAX(TIME) AS end_time,
    MIN(LONGITUDE) AS min_longitude,
    MAX(LONGITUDE) AS max_longitude,
    MIN(LATITUDE) AS min_latitude,
    MAX(LATITUDE) AS max_latitude,
    COUNT(*)  AS n_rows
    FROM read_parquet(
    '{s3_uri}**/*.parquet',
    hive_partitioning=1
    )
    WHERE {partition_column_name} IN ({partition_names_sql})
    GROUP BY {partition_column_name}
    ORDER BY {partition_column_name}
    """
    
else:
    
    # This is not an IMOS dataset, so just get row counts by partition
    # Variable names will likely vary by dataset
    # (e.g. argo)
    
    sql = f"""
    SELECT
    {partition_column_name},
    COUNT(*)  AS n_rows
    FROM read_parquet(
    '{s3_uri}**/*.parquet',
    hive_partitioning=1
    )
    WHERE {partition_column_name} IN ({partition_names_sql})
    GROUP BY {partition_column_name}
    ORDER BY {partition_column_name}
    """
    

In [13]:
stats = con.execute(sql).fetchdf()

# Testing 
# hive partitioning, LIMIT 1, group/order by deployment code, count rows, min/max time = 6m45s
# Specifying a list of partition names decreased time to ~5m

100% ▕██████████████████████████████████████▏ (00:02:37.14 elapsed)       


In [14]:

if 'start_time' in stats.columns and 'end_time' in stats.columns:
    # Format start_time and end_time as strings for better display in HTML table
    stats["start_time"] = stats["start_time"].dt.strftime("%Y-%m-%d %H:%M")
    stats["end_time"]   = stats["end_time"].dt.strftime("%Y-%m-%d %H:%M")

stats

,timestamp,start_time,end_time,min_longitude,max_longitude,min_latitude,max_latitude,n_rows
0,1199145600,2008-01-11 09:32,2008-01-31 19:18,144.458000,149.440000,-45.800300,-42.891100,16705
1,1201824000,2008-02-04 03:06,2008-02-25 20:00,134.217000,147.339000,-43.692100,-34.844000,20201
2,1204329600,2008-03-21 06:11,2008-03-25 09:13,133.669000,161.171000,-46.930500,-34.926900,12353
3,1207008000,2008-04-05 04:34,2008-04-28 16:46,145.564000,170.341000,-44.810600,-21.918100,14544
4,1209600000,NaN,NaN,-179.999000,180.000000,-22.326300,-13.756400,21182
...,...,...,...,...,...,...,...,...
188,1767225600,2026-01-02 05:50,2026-01-31 23:59,143.156426,153.290537,-65.798529,-42.881587,30365
189,1769904000,2026-02-01 00:01,2026-02-28 23:59,131.767556,152.367026,-66.623483,-42.975344,27191
190,1772323200,2026-03-01 00:01,2026-03-31 23:58,112.416009,147.507340,-59.304903,-16.421641,18926
191,1775001600,2026-04-01 00:00,2026-04-30 23:59,112.023363,138.860497,-36.870336,-21.545208,16822


In [15]:
# Get total number of rows across all partitions
total_rows = stats["n_rows"].sum()
print(total_rows)

3680653


In [16]:
# Create a figure to plot the approximate locations of the partitions based on their min latitude and longitude.

fig = plt.figure(figsize=(8, 8))
ax = plt.axes(projection=ccrs.PlateCarree())
# Set Australia extent
ax.set_extent([110, 155, -45, -10], crs=ccrs.PlateCarree())
# Add coastline
ax.coastlines(resolution='10m')
ax.add_feature(cfeature.BORDERS, linewidth=0.5)
if 'min_latitude' in stats.columns and 'min_longitude' in stats.columns:
    # Scatter
    ax.scatter(
        stats['min_longitude'],
        stats['min_latitude'],
        s=stats['n_rows'] / 300,
        edgecolors='k',
        transform=ccrs.PlateCarree()
    )
# Set title
ax.set_title("Partition approx locations (Australia)")

# Save figure to memory buffer (to include in the html summary)
buf = io.BytesIO()
plt.savefig(buf, format="png", bbox_inches="tight")
plt.close(fig)
buf.seek(0)
# Convert to base64
img_base64 = base64.b64encode(buf.read()).decode("utf-8")


### Export information to a html 

In [17]:
def capture_prints_to_html(func, dataset_name, output_file='output.html', *args, **kwargs):
    """
    Execute a function and capture all print statements to an HTML file.
    
    Parameters:
    -----------
    func : callable
        The function to execute
    dataset_name : str
        Name of the dataset
    output_file : str
        Path to the output HTML file
    *args, **kwargs
        Arguments to pass to the function
    
    Returns:
    --------
    result : any
    func : callable
        The function to execute
    output_file : str
        Path to the output HTML file
    *args, **kwargs
        Arguments to pass to the function
        }}
        .container {{
            background-color: white;
            padding: 30px;
            border-radius: 8px;
            box-shadow: 0 2px 4px rgba(0,0,0,0.1);
        }}
        h1 {{
            color: #333;vessel_fishsoop_real
    
    Returns:
    --------
    result : any
        The return value of the function
    """
    # Create a StringIO object to capture stdout
    captured_output = StringIO()
    old_stdout = sys.stdout
    
    try:
        # Redirect stdout to our StringIO object
        sys.stdout = captured_output
        
        # Execute the function
        result = func(*args, **kwargs)
        
    finally:
        # Restore stdout
        sys.stdout = old_stdout
    
    # Get the captured output
    output_text = captured_output.getvalue()
    
    # Create HTML content
    html_content = f"""<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>{dataset_name}</title>
    <style>
        body {{
            font-family: 'Courier New', monospace;
            font-size: 16px;
            background-color: #f5f5f5;
            padding: 20px;
            max-width: 95vw;
            margin: 0 auto;
        }}
        .container {{
            background-color: white;
            padding: 30px;
            border-radius: 8px;
            box-shadow: 0 2px 4px rgba(0,0,0,0.1);
        }}
        h1 {{
            color: #333;
            font-size: 28px;
            border-bottom: 2px solid #4CAF50;
            padding-bottom: 10px;
        }}
        .timestamp {{
            color: #666;
            font-size: 14px;
            margin-bottom: 20px;
        }}
        pre {{
            background-color: #f8f8f8;
            border: 1px solid #ddd;
            border-radius: 4px;
            padding: 15px;
            overflow-x: auto;
            line-height: 1.6;
            font-size: 15px;
        }}
    </style>
</head>
<body>
    <div class="container">
        <h1>{dataset_name}</h1>
        <div class="timestamp">Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}</div>
        <pre>{output_text}</pre>
        <img src="data:image/png;base64,{img_base64}" style="max-width:100%; height:auto;" />
    </div>
</body>
</html>"""
    
    # Write to file
    with open(output_file, 'w', encoding='utf-8') as f:
        f.write(html_content)
    
    print(f"Output saved to: {output_file}")
    
    return result


In [ ]:
def add_title(title_str):
    print("=" * 150)
    print(title_str.center(150))
    print("=" * 150)
    print(' ' * 150)


def html_summary_content():
    
    # S3 summaryTotal notebook runtimep is much faste
    add_title("PARQUET FILE SUMMARY")
    _,_,files = get_s3_size_and_file_count(s3_uri)
    print(' ' * 150)
    print(f"Parquet file created: {first}")
    print(f"Parquet file last updated: {latest}")
    print(' ' * 150)

    # List the files in the dataset
    add_title("DATASET VARIABLES AND PARTITIONS")
    summarise_ds(s3_uri)
    print(' ' * 150)
    
    # Deployment stats
    add_title("DATASET STATISTICS")
    print(f"\nTotal rows: {total_rows}")
    print(' ' * 150)
    print(stats.to_string())
    print(' ' * 150)
    
    # List the files in the dataset
    add_title("PARQUET FILES IN DATASET")
    print(' ' * 150)
    for file in files:
        print(file)
    print("=" * 150)
    print(' ' * 150)
    add_title("NETCDF FILES APPENDED")
    print(' ' * 150)
    for file in nc_files:
        print(file)
    print("=" * 150)
    print(' ' * 150)

    # work out total notebook runtime
    notebook_end = time.perf_counter()
    total_elapsed = notebook_end - notebook_start
    minutes = int(total_elapsed // 60)
    seconds = total_elapsed % 60
    print(' ' * 150)
    print(f"Total notebook runtime: {minutes}m {seconds:.1f}s")
    print(' ' * 150)
    

# Capture all output to a single HTML file
capture_prints_to_html(html_summary_content, dataset_name, f'{dataset_name}_summary_nonprod.html')


Output saved to: vessel_co2_delayed_qc_summary.html


### Notes

- The use of the DuckDB progress bar was a quick way to see if my query was reasonable (if it would take too long)

- There are two progress bars when loading into memory (first one for the DuckDB query, and second one for memory (much longer))
- it would estimate a longer period of time to execute, and then suddenly finish (e.g. 35 min at first, took 18 mins)
- Earlier testing suggests using duckdb with 14 cores makes querying twice as fast
- Setting hive partitioning to True and enabling partition pruning is much faster for getting statistics - perhaps because DuckDB is told where to look rather than discovering partitions/loading metadata?
- Multiple ways to get partition information
- Even though `slocum_glider_delayed_qc.parquet` is much larger (~34.2GB) compared to `argo.parquet` (~3GB), the notebook required more time to complete, suggesting that we need to optimise this file. Other similar sized datasets took ~7mins instead of ~25mins. 
- FishSOOP only ~ 0.5GB  but >51000 partitions, so incredibly slow to analyse! initial estimate > 3 hours to run the notebook, same for AODN WAVE RT (~ 0.2GB, > 27000 partitions, did 11% in 3 mins)